# BLIP-1 Explorer — LoRA Student Training (Knowledge Distillation)

**Student model:** [Salesforce/blip-image-captioning-large](https://huggingface.co/Salesforce/blip-image-captioning-large) (~400 M params, 9× smaller than teacher)  
**Dataset:** [kevintang2048/underwater-image-captioning-dataset-uicd](https://www.kaggle.com/datasets/kevintang2048/underwater-image-captioning-dataset-uicd)  
**Accelerator:** Kaggle Notebooks — T4 x2

---

## Prerequisites

Attach the `kevintang2048/blip2-explorer-pseudo-labels` dataset as a Kaggle input. This notebook loads `pseudo_labels.json` directly and does not re-run the teacher.

---

## Strategy

Fine-tune BLIP-1 Large with LoRA (r=32) on teacher pseudo-labels, preserving the CLIP auxiliary loss adapted for BLIP-1's decoder hidden states.

Expected composite score: **80–88% of teacher baseline** (CLIP score protected by shared ViT-L/16 encoder).

---

## Notebook Structure

| # | Section |
|---|--------|
| 1 | Environment Setup |
| 2 | GPU / Memory Check |
| 3 | Configuration |
| 4 | Dataset Loading |
| 5 | Train / Val / Test Split |
| 6 | Load Pseudo-labels |
| 7 | PyTorch Dataset & DataLoaders |
| 8 | Load Student Model & Processor |
| 9 | Verify LoRA Target Modules |
| 10 | Apply LoRA (PEFT) |
| 11 | CLIP Auxiliary Setup |
| 12 | Optimizer & Scheduler |
| 13 | Training Loop |
| 14 | Loss Visualisation |
| 15 | Save Outputs |
| 16 | Package Outputs as ZIP |

## 1 · Environment Setup

In [ ]:

# ── Install packages not pre-installed in Kaggle Notebooks ──────────────────
import subprocess, sys

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "peft>=0.5.0",
    "pycocoevalcap",
    "pycocotools",
    "open_clip_torch",
], check=True)

# ── Standard library ─────────────────────────────────────────────────────────
import gc
import json
import os
import random
import time
from pathlib import Path

# ── Third-party ───────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from PIL import Image
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

# ── HuggingFace ───────────────────────────────────────────────────────────────
from transformers import (
    AutoProcessor,
    Blip2ForConditionalGeneration,   # teacher (BLIP-2 OPT-2.7B)
    BlipConfig,
    BlipForConditionalGeneration,    # student (BLIP-1 Large)
    BlipProcessor,
    get_linear_schedule_with_warmup,
)

# ── PEFT / LoRA ───────────────────────────────────────────────────────────────
from peft import LoraConfig, get_peft_model, TaskType

print("All imports successful.")


## 2 · GPU / Memory Check

In [ ]:
gc.collect()
torch.cuda.empty_cache()

if not torch.cuda.is_available():
    raise EnvironmentError("No CUDA device found — enable the T4 x2 accelerator in Kaggle settings.")

device = torch.device("cuda")
n_gpus = torch.cuda.device_count()
print(f"Number of GPUs: {n_gpus}")
for i in range(n_gpus):
    props = torch.cuda.get_device_properties(i)
    total_vram = props.total_memory / 1024**3
    free_vram  = (props.total_memory - torch.cuda.memory_allocated(i)) / 1024**3
    print(f"  GPU {i}: {props.name}  |  Total VRAM: {total_vram:.1f} GB  |  Free VRAM: {free_vram:.1f} GB")

## 3 · Configuration

In [ ]:
CONFIG = {
    # ── Student model ─────────────────────────────────────────────────────────
    "student_model_id": "Salesforce/blip-image-captioning-large",

    # ── Teacher (fine-tuned BLIP-2 merged model) ──────────────────────────────
    "teacher_path": "/kaggle/input/models/kevintang2048/blip2-explorer/pytorch/default/2/blip2_lora_uicd/merged_model",
    # Processor subfolder exists but image preprocessor config is absent;
    # the cell falls back to the original model ID when loading fails.
    "teacher_processor_path": "/kaggle/input/models/kevintang2048/blip2-explorer/pytorch/default/2/blip2_lora_uicd/processor",

    # ── Dataset ───────────────────────────────────────────────────────────────
    "dataset_root": "/kaggle/input/datasets/kevintang2048/underwater-image-captioning-dataset-uicd",
    "split": (0.8, 0.1, 0.1),   # train / val / test fractions
    "seed": 42,

    # ── Pseudo-label generation ───────────────────────────────────────────────
    # 1 beam-search caption + (n_pseudo_captions - 1) sampled captions per image
    "n_pseudo_captions": 5,
    "pseudo_beam_width": 5,
    "pseudo_temperature": 0.7,
    "pseudo_top_p": 0.9,
    "pseudo_max_new_tokens": 40,
    "pseudo_label_path": "/kaggle/input/datasets/kevintang2048/blip2-explorer-pseudo-labels/pseudo_labels.json",

    # ── Training ──────────────────────────────────────────────────────────────
    "epochs": 5,
    "batch_size": 4,             # slightly larger than teacher — student is smaller
    "lr": 2e-4,
    "max_length": 256,
    "warmup_ratio": 0.1,

    # ── CLIP auxiliary objective ──────────────────────────────────────────────
    # Raised from 0.2 (teacher) — student needs stronger CLIP alignment
    "clip_aux_weight": 0.35,
    "clip_aux_warmup_epochs": 1,
    "clip_aux_every_n_steps": 5,

    # ── Metric-aware validation policy ────────────────────────────────────────
    "full_metrics_every_n_epochs": 1,
    "metrics_subset_size": 128,
    "max_new_tokens_eval": 40,
    "num_beams_eval": 3,

    # Composite score weights — same as teacher for fair comparison
    "metric_weights": {
        "clip":   0.50,
        "cider":  0.25,
        "meteor": 0.10,
        "bleu4":  0.08,
        "rouge":  0.07,
    },
    "metric_norm": {
        "clip":   1.0,
        "cider":  2.0,
        "meteor": 1.0,
        "bleu4":  1.0,
        "rouge":  1.0,
    },

    # ── LoRA ──────────────────────────────────────────────────────────────────
    # Higher rank than teacher (16) to compensate for smaller model capacity
    "lora_r": 32,
    "lora_alpha": 32,
    "lora_dropout": 0.1,
    "lora_bias": "none",
    # BLIP-1 uses a BERT-based text decoder whose attention Linear layers are
    # named "query", "key", "value" (not "q_proj"/"k_proj"/"v_proj" like OPT).
    # Verified via named_modules() scan in Section 9 before applying LoRA.
    "lora_target_modules": ["query", "key", "value"],

    # ── Output ────────────────────────────────────────────────────────────────
    "output_dir": "/kaggle/working/blip1_student_uicd",
}

# Create output directory
Path(CONFIG["output_dir"]).mkdir(parents=True, exist_ok=True)

# Deterministic seeding
random.seed(CONFIG["seed"])
np.random.seed(CONFIG["seed"])
torch.manual_seed(CONFIG["seed"])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CONFIG["seed"])

print("CONFIG:")
for k, v in CONFIG.items():
    print(f"  {k}: {v}")

## 4 · Dataset Loading

Parses `UIC-captions.txt` (format: `<image_id>#<caption_index> <caption_text>`) and pairs each image filename with its list of reference captions. Identical to the teacher notebook.

In [ ]:
def load_captions(captions_path: Path) -> dict:
    """Parse UIC-captions.txt into {filename: [caption, ...]}."""
    captions_dict = {}
    with open(captions_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            # Format: <img_id>#<n> <caption text>
            parts = line.split(" ", 1)
            if len(parts) != 2:
                continue
            img_key, caption = parts
            filename = img_key.split("#")[0]
            captions_dict.setdefault(filename, []).append(caption.strip())
    return captions_dict


dataset_root = Path(CONFIG["dataset_root"])

captions_file = next(dataset_root.rglob("UIC-captions.txt"), None)
if captions_file is None:
    raise FileNotFoundError(f"UIC-captions.txt not found under {dataset_root}")

dataset_base = captions_file.parent
image_dir    = dataset_base / "uic_224x224_image"
if not image_dir.exists():
    image_dirs = [d for d in dataset_base.iterdir() if d.is_dir()]
    if not image_dirs:
        raise FileNotFoundError(f"No image directory found under {dataset_base}")
    image_dir = image_dirs[0]

captions_dict = load_captions(captions_file)

dataset = []
for filename, captions in captions_dict.items():
    img_path = image_dir / filename
    if img_path.exists():
        dataset.append({
            "image_path":     str(img_path),
            "image_filename": filename,
            "captions":       captions,
        })

print(f"Captions file  : {captions_file}")
print(f"Image directory: {image_dir}")
print(f"Total samples  : {len(dataset)}")
print(f"Sample entry   : {dataset[0]}")

## 5 · Train / Val / Test Split

Uses the same 80/10/10 split with `seed=42` as the teacher notebook so evaluation sets are directly comparable.

In [ ]:
train_frac, val_frac, test_frac = CONFIG["split"]

dataset_sorted = sorted(dataset, key=lambda x: x["image_filename"])
rng = random.Random(CONFIG["seed"])
rng.shuffle(dataset_sorted)

n = len(dataset_sorted)
n_train = int(n * train_frac)
n_val   = int(n * val_frac)

train_set = dataset_sorted[:n_train]
val_set   = dataset_sorted[n_train : n_train + n_val]
test_set  = dataset_sorted[n_train + n_val :]

print(f"Total : {n}")
print(f"Train : {len(train_set)}  ({len(train_set)/n*100:.1f}%)")
print(f"Val   : {len(val_set)}   ({len(val_set)/n*100:.1f}%)")
print(f"Test  : {len(test_set)}  ({len(test_set)/n*100:.1f}%)")

## 6 · Load Pseudo-labels

Loads `pseudo_labels.json` from the `kevintang2048/blip2-explorer-pseudo-labels` Kaggle input dataset. Raises an error if the file is missing — check that the dataset is attached as a Kaggle input.

In [ ]:
pseudo_label_path = Path(CONFIG["pseudo_label_path"])

if not pseudo_label_path.exists():
    raise FileNotFoundError(
        f"Pseudo-labels not found at {pseudo_label_path}. "
        "Add kevintang2048/blip2-explorer-pseudo-labels as a Kaggle input dataset."
    )

with open(pseudo_label_path, "r") as f:
    pseudo_labels = json.load(f)

print(f"Loaded {len(pseudo_labels)} pseudo-label entries ({CONFIG['n_pseudo_captions']} captions each).")

## 7 · PyTorch Dataset & DataLoaders

`PseudoLabelDataset` loads `pseudo_labels.json` and flattens it: each of the N images × 5 pseudo-captions becomes a separate training pair, giving the student 5× more supervision signal than the teacher had.  Val and test loaders use the **original** UICD reference captions (not pseudo-labels) so that evaluation is against the true ground truth.

In [ ]:
class PseudoLabelDataset(Dataset):
    """Training dataset built from teacher pseudo-labels.

    Flattens the N×K pseudo-label structure into N*K individual (image, caption)
    pairs — the student sees every caption as a separate training example.
    """

    def __init__(self, pseudo_labels: list, processor, max_length: int = 256,
                 image_filenames: set | None = None):
        """
        Args:
            pseudo_labels:     Full list loaded from pseudo_labels.json.
            processor:         BlipProcessor instance.
            max_length:        Tokenizer max sequence length.
            image_filenames:   If provided, restrict to images in this set
                               (e.g. only images from the train split).
        """
        self.processor  = processor
        self.max_length = max_length
        self.pairs      = []  # list of (image_path, caption_str)

        for entry in pseudo_labels:
            if image_filenames is not None and entry["image_filename"] not in image_filenames:
                continue
            for cap in entry["captions"]:
                cap = cap.strip()
                if cap:
                    self.pairs.append((entry["image_path"], cap))

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        image_path, caption = self.pairs[idx]
        image = Image.open(image_path).convert("RGB")

        encoding = self.processor(
            images=image,
            text=caption,
            return_tensors="pt",
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
        )

        pixel_values   = encoding["pixel_values"].squeeze(0)
        input_ids      = encoding["input_ids"].squeeze(0)
        attention_mask = encoding["attention_mask"].squeeze(0)

        labels = input_ids.clone()
        labels[labels == self.processor.tokenizer.pad_token_id] = -100

        return {
            "pixel_values":   pixel_values,
            "input_ids":      input_ids,
            "attention_mask": attention_mask,
            "labels":         labels,
        }


class RefCaptionDataset(Dataset):
    """Eval dataset using original UICD reference captions (not pseudo-labels).

    Used for val / test DataLoaders so evaluation is against ground truth.
    Randomly picks one reference caption per item, matching the teacher's eval setup.
    """

    def __init__(self, samples: list, processor, max_length: int = 256):
        self.samples    = samples
        self.processor  = processor
        self.max_length = max_length

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        item    = self.samples[idx]
        image   = Image.open(item["image_path"]).convert("RGB")
        caption = random.choice(item["captions"])

        encoding = self.processor(
            images=image,
            text=caption,
            return_tensors="pt",
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
        )

        pixel_values   = encoding["pixel_values"].squeeze(0)
        input_ids      = encoding["input_ids"].squeeze(0)
        attention_mask = encoding["attention_mask"].squeeze(0)

        labels = input_ids.clone()
        labels[labels == self.processor.tokenizer.pad_token_id] = -100

        return {
            "pixel_values":   pixel_values,
            "input_ids":      input_ids,
            "attention_mask": attention_mask,
            "labels":         labels,
        }


# DataLoaders are created after the student model & processor are loaded (Section 8).
print("PseudoLabelDataset and RefCaptionDataset classes defined.")

## 8 · Load Student Model & Processor

Loads `Salesforce/blip-image-captioning-large` in **fp16** with `device_map="auto"`. All base parameters are frozen — only LoRA adapters (added in Section 10) will be trainable.

In [ ]:

import warnings
warnings.filterwarnings("ignore", message=".*query_tokens.*")

gc.collect()
torch.cuda.empty_cache()

if not torch.cuda.is_available():
    raise EnvironmentError("No CUDA device found — enable the T4 x2 accelerator in Kaggle settings.")

device = torch.device("cuda")
n_gpus = torch.cuda.device_count()
print(f"Number of GPUs: {n_gpus}")
for i in range(n_gpus):
    props = torch.cuda.get_device_properties(i)
    total_vram = props.total_memory / 1024**3
    free_vram  = (props.total_memory - torch.cuda.memory_allocated(i)) / 1024**3
    print(f"  GPU {i}: {props.name}  |  Total VRAM: {total_vram:.1f} GB  |  Free VRAM: {free_vram:.1f} GB")

student_model_id = CONFIG["student_model_id"]

# ── Processor ─────────────────────────────────────────────────────────────────
print("Loading student processor...")
processor = BlipProcessor.from_pretrained(student_model_id)
processor.tokenizer.model_max_length = CONFIG["max_length"]

# ── Config: disable weight tying to avoid the tied-weights warning ─────────────
# The checkpoint stores both word_embeddings and predictions.decoder weights
# separately, so tying them is unnecessary and triggers a noisy warning.
# Must be set at BOTH the top-level BlipConfig AND text_config — the model
# loading code checks the top-level flag; text_config is set for completeness.
student_config = BlipConfig.from_pretrained(student_model_id)
student_config.tie_word_embeddings = False
student_config.text_config.tie_word_embeddings = False

# ── Base model in fp16 ────────────────────────────────────────────────────────
print("Loading student base model (fp16)...")
model = BlipForConditionalGeneration.from_pretrained(
    student_model_id,
    config=student_config,
    torch_dtype=torch.float16,
    device_map="auto",
)

# Freeze all base parameters
for param in model.parameters():
    param.requires_grad = False

total_params = sum(p.numel() for p in model.parameters())
print(f"\nStudent model loaded.")
print(f"Total parameters : {total_params / 1e6:.1f} M")
print(f"Device map       : {model.hf_device_map if hasattr(model, 'hf_device_map') else 'single device'}")

# ── Build DataLoaders now that processor is available ─────────────────────────
train_filenames = {s["image_filename"] for s in train_set}

train_dataset = PseudoLabelDataset(pseudo_labels, processor, CONFIG["max_length"],
                                   image_filenames=train_filenames)
val_dataset   = RefCaptionDataset(val_set,  processor, CONFIG["max_length"])
test_dataset  = RefCaptionDataset(test_set, processor, CONFIG["max_length"])

train_loader = DataLoader(train_dataset, batch_size=CONFIG["batch_size"], shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_dataset,   batch_size=CONFIG["batch_size"], shuffle=False, num_workers=0)
test_loader  = DataLoader(test_dataset,  batch_size=1,                   shuffle=False, num_workers=0)

print(f"\nTrain pairs (N×{CONFIG['n_pseudo_captions']}) : {len(train_dataset)}")
print(f"Train batches : {len(train_loader)}")
print(f"Val batches   : {len(val_loader)}")
print(f"Test batches  : {len(test_loader)}")


## 9 · Verify LoRA Target Modules

BLIP-1's text decoder is BERT-based and uses `query` / `key` / `value` as the `nn.Linear` attribute names — different from the teacher's OPT decoder which used `q_proj` / `k_proj` / `v_proj`. This cell scans all `nn.Linear` submodules and confirms the names before applying LoRA, so the config never silently targets zero layers.

In [ ]:
# Collect the local attribute name (last component) of every nn.Linear in the model.
# Duplicates are collapsed into a set.  We look for "query"/"key"/"value" from the
# BERT text decoder and optionally "q_proj"/"k_proj"/"v_proj" from the ViT visual encoder.
linear_leaf_names = sorted({
    name.split(".")[-1]
    for name, module in model.named_modules()
    if isinstance(module, nn.Linear)
})
print("Unique nn.Linear leaf names in student model:")
for n in linear_leaf_names:
    print(f"  {n}")

# Verify the configured targets are actually present
configured_targets = set(CONFIG["lora_target_modules"])
found_in_model = configured_targets & set(linear_leaf_names)
missing = configured_targets - set(linear_leaf_names)

print(f"\nConfigured lora_target_modules : {sorted(configured_targets)}")
print(f"Found in model                 : {sorted(found_in_model)}")
if missing:
    print(f"WARNING — NOT FOUND            : {sorted(missing)}")
    print("Update CONFIG['lora_target_modules'] before proceeding to Section 10.")
else:
    print("All target modules verified ✓")


## 10 · Apply LoRA (PEFT)

Injects low-rank adapter matrices (r=32, α=32) into the `query`, `key`, and `value` projections of the BLIP-1 text decoder. The higher rank vs. the teacher (16) compensates for the smaller model capacity when absorbing domain knowledge.

LoRA is applied to **`model.text_decoder` only** (not the full `BlipForConditionalGeneration`). Wrapping the whole model causes PEFT's `LoraModel` to intercept every child module's `__call__`, which creates a conflicting `inputs_embeds` positional/keyword argument collision deep inside `BlipEncoder`. Scoping PEFT to the text decoder keeps the visual encoder completely outside PEFT's forward-pass bookkeeping.

Saving uses `model.text_decoder.save_pretrained()` for adapter-only checkpoints, and `model.text_decoder = model.text_decoder.merge_and_unload()` followed by `model.save_pretrained()` for the final merged artefact (Section 15).


In [ ]:

lora_config = LoraConfig(
    task_type=TaskType.FEATURE_EXTRACTION,
    r=CONFIG["lora_r"],
    lora_alpha=CONFIG["lora_alpha"],
    lora_dropout=CONFIG["lora_dropout"],
    bias=CONFIG["lora_bias"],
    target_modules=CONFIG["lora_target_modules"],
)

# Wrap ONLY the text decoder — keeps BlipEncoder fully outside PEFT's scope,
# preventing the inputs_embeds positional/keyword collision in the visual path.
model.text_decoder = get_peft_model(model.text_decoder, lora_config)

# ── Trainable parameter report ────────────────────────────────────────────────
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
all_params        = sum(p.numel() for p in model.parameters())
pct               = 100 * trainable_params / all_params

print(f"Trainable parameters : {trainable_params:,}  ({pct:.2f}% of total)")
print(f"Total parameters     : {all_params:,}")
model.text_decoder.print_trainable_parameters()


## 11 · CLIP Auxiliary Setup & Metric Helpers

**Key adaptation from teacher:** BLIP-1 has no `qformer_outputs`. Instead, the decoder hidden states are pooled and projected into CLIP embedding space for the auxiliary alignment loss. The projector input dimension is still 768 (BLIP-1's text decoder hidden size matches BLIP-2's Q-Former hidden size).

All other metric helpers (`compute_clip_similarity_on_samples`, `compute_text_metrics_on_samples`, `compute_composite_score`) are identical to the teacher notebook.

In [ ]:
import open_clip

from pycocoevalcap.bleu.bleu import Bleu
from pycocoevalcap.cider.cider import Cider
from pycocoevalcap.meteor.meteor import Meteor
from pycocoevalcap.rouge.rouge import Rouge


# ── CLIP evaluation model (frozen, used for aux loss + eval metric) ───────────
clip_metric_model, _, clip_metric_preprocess = open_clip.create_model_and_transforms(
    "ViT-B-32", pretrained="laion2b_s34b_b79k"
)
clip_metric_model = clip_metric_model.to(device).eval()
clip_metric_model.requires_grad_(False)
clip_metric_tokenizer = open_clip.get_tokenizer("ViT-B-32")

# CLIP normalisation constants (used when re-normalising pixel_values for CLIP)
CLIP_MEAN = torch.tensor([0.48145466, 0.4578275,  0.40821073], device=device).view(1, 3, 1, 1)
CLIP_STD  = torch.tensor([0.26862954, 0.26130258, 0.27577711], device=device).view(1, 3, 1, 1)

# ── Determine CLIP embedding dimension ───────────────────────────────────────
with torch.no_grad():
    _dummy_txt = clip_metric_tokenizer(["underwater scene"]).to(device)
    clip_embed_dim = int(clip_metric_model.encode_text(_dummy_txt).shape[-1])

# ── CLIP projector: text-decoder hidden → CLIP embedding space ───────────────
# BLIP-1 text decoder hidden size = 768 (same as BLIP-2 Q-Former hidden size)
# so no dimension change is needed relative to the teacher notebook.
student_hidden_dim = 768
clip_projector = nn.Linear(student_hidden_dim, clip_embed_dim, bias=False).to(device)
print(f"CLIP projector : text_hidden={student_hidden_dim} → clip_dim={clip_embed_dim}")


# ── Image re-normalisation helper ────────────────────────────────────────────
def _prepare_clip_images_from_blip_pixels(pixel_values: torch.Tensor, proc) -> torch.Tensor:
    """Convert BLIP-normalised pixel_values to CLIP-normalised tensors.

    BLIP-1 uses different image_mean/image_std than BLIP-2; this function reads
    them dynamically from the processor so the denormalisation is always correct.
    """
    mean = torch.tensor(proc.image_processor.image_mean,
                        device=pixel_values.device).view(1, 3, 1, 1)
    std  = torch.tensor(proc.image_processor.image_std,
                        device=pixel_values.device).view(1, 3, 1, 1)
    x = pixel_values.float() * std + mean          # undo BLIP normalisation
    x = x.clamp(0.0, 1.0)
    x = nn.functional.interpolate(x, size=(224, 224), mode="bicubic", align_corners=False)
    x = (x - CLIP_MEAN) / CLIP_STD                 # apply CLIP normalisation
    return x.to(dtype=torch.float16)


# ── CLIP auxiliary loss (BLIP-1 adaptation) ───────────────────────────────────
def _extract_decoder_hidden_for_clip(outputs) -> torch.Tensor | None:
    """Extract decoder hidden states from BLIP-1 forward outputs.

    BLIP-1 (BlipForConditionalGeneration) does NOT expose qformer_outputs.
    Instead, pool the last decoder hidden-state layer — its hidden size (768)
    matches the teacher's Q-Former hidden size, so clip_projector is unchanged.

    Requires the forward call to be made with output_hidden_states=True.
    """
    # outputs.decoder_hidden_states: tuple of (batch, seq_len, 768) per layer
    dec_hs = getattr(outputs, "decoder_hidden_states", None)
    if dec_hs is None:
        return None
    last_layer = dec_hs[-1]   # (B, seq_len, 768)
    return last_layer


def compute_clip_aux_loss(outputs, pixel_values: torch.Tensor, proc) -> torch.Tensor:
    dec_hidden = _extract_decoder_hidden_for_clip(outputs)
    if dec_hidden is None:
        return torch.tensor(0.0, device=pixel_values.device, dtype=torch.float32)

    pooled   = dec_hidden.mean(dim=1).float()                    # (B, 768)
    txt_feat = clip_projector(pooled)
    txt_feat = txt_feat / txt_feat.norm(dim=-1, keepdim=True).clamp_min(1e-6)

    clip_imgs = _prepare_clip_images_from_blip_pixels(pixel_values, proc)
    with torch.no_grad():
        img_feat = clip_metric_model.encode_image(clip_imgs).float()
        img_feat = img_feat / img_feat.norm(dim=-1, keepdim=True).clamp_min(1e-6)

    cosine_sim = (txt_feat * img_feat).sum(dim=-1)
    return (1.0 - cosine_sim).mean()


# ── Evaluation metric helpers (identical to teacher notebook) ─────────────────
def _deterministic_subset(samples: list, max_items: int, seed: int) -> list:
    if max_items <= 0 or len(samples) <= max_items:
        return samples
    rng = random.Random(seed)
    idx = list(range(len(samples)))
    rng.shuffle(idx)
    return [samples[i] for i in idx[:max_items]]


def _clean_caption(text: str) -> str:
    return " ".join((text or "").strip().split())


def compute_clip_similarity_on_samples(
    m, proc, samples: list, max_new_tokens: int, num_beams: int
) -> float:
    if not samples:
        return 0.0
    sims = []
    with torch.no_grad():
        for item in tqdm(samples, desc="CLIP eval", leave=False):
            image   = Image.open(item["image_path"]).convert("RGB")
            inputs  = proc(images=image, return_tensors="pt").to(device)
            gen_ids = m.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                num_beams=num_beams,
                do_sample=False,
                early_stopping=True,
            )
            pred    = _clean_caption(proc.batch_decode(gen_ids, skip_special_tokens=True)[0])

            img_t   = clip_metric_preprocess(image).unsqueeze(0).to(device)
            txt_t   = clip_metric_tokenizer([pred]).to(device)
            i_feat  = clip_metric_model.encode_image(img_t)
            t_feat  = clip_metric_model.encode_text(txt_t)
            i_feat  = i_feat / i_feat.norm(dim=-1, keepdim=True)
            t_feat  = t_feat / t_feat.norm(dim=-1, keepdim=True)
            sims.append(float((i_feat @ t_feat.T).squeeze().item()))
    return float(np.mean(sims)) if sims else 0.0


def compute_text_metrics_on_samples(
    m, proc, samples: list, max_new_tokens: int, num_beams: int
) -> dict:
    empty = {"cider": 0.0, "meteor": 0.0, "bleu1": 0.0,
             "bleu2": 0.0, "bleu3": 0.0, "bleu4": 0.0, "rouge": 0.0}
    if not samples:
        return empty
    gts, res = {}, {}
    with torch.no_grad():
        for i, item in enumerate(tqdm(samples, desc="COCO eval", leave=False)):
            image   = Image.open(item["image_path"]).convert("RGB")
            inputs  = proc(images=image, return_tensors="pt").to(device)
            gen_ids = m.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                num_beams=num_beams,
                do_sample=False,
                early_stopping=True,
            )
            pred = _clean_caption(proc.batch_decode(gen_ids, skip_special_tokens=True)[0])
            img_id       = f"img_{i}"
            gts[img_id]  = [_clean_caption(c) for c in item["captions"]]
            res[img_id]  = [pred]

    cider_score,  _ = Cider().compute_score(gts, res)
    meteor_score, _ = Meteor().compute_score(gts, res)
    bleu_scores,  _ = Bleu(4).compute_score(gts, res)
    rouge_score,  _ = Rouge().compute_score(gts, res)
    return {
        "cider":  float(cider_score),
        "meteor": float(meteor_score),
        "bleu1":  float(bleu_scores[0]),
        "bleu2":  float(bleu_scores[1]),
        "bleu3":  float(bleu_scores[2]),
        "bleu4":  float(bleu_scores[3]),
        "rouge":  float(rouge_score),
    }


def compute_composite_score(metrics: dict, weights: dict, norm: dict) -> float:
    weighted_sum, total_weight = 0.0, 0.0
    for key, w in weights.items():
        v = metrics.get(key, float("nan"))
        if v is None or not np.isfinite(v):
            continue
        d = max(float(norm.get(key, 1.0)), 1e-8)
        weighted_sum += w * (float(v) / d)
        total_weight += w
    return weighted_sum / total_weight if total_weight > 0 else -1.0


print("CLIP auxiliary loss + metric helpers ready.")

## 12 · Optimizer & Scheduler

In [ ]:
# LoRA adapter params + CLIP projector params are both trained
trainable_params = list(filter(lambda p: p.requires_grad, model.parameters()))
trainable_params += list(clip_projector.parameters())

optimizer = AdamW(trainable_params, lr=CONFIG["lr"])

total_training_steps = CONFIG["epochs"] * len(train_loader)
warmup_steps         = int(total_training_steps * CONFIG["warmup_ratio"])

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_training_steps,
)

print(f"Total training steps : {total_training_steps}")
print(f"Warmup steps         : {warmup_steps}")
print(f"Learning rate        : {CONFIG['lr']}")
print(f"Optimiser param groups: {len(optimizer.param_groups)}")

## 13 · Training Loop

Training optimises a hybrid objective: **CE loss + warm-started CLIP auxiliary alignment loss** (weight 0.35, higher than teacher's 0.05 to compensate for smaller model capacity).

Validation computes CE every epoch, CLIP similarity every epoch, and full COCO metrics on a periodic schedule. Best checkpoint selection uses the same weighted composite score (CLIP-prioritised) as the teacher for a fair comparison.

In [ ]:

# ── Tracking ──────────────────────────────────────────────────────────────────
step_train_losses = []   # list of (global_step, total_loss)
epoch_val_losses  = []   # list of val_loss per epoch
epoch_boundaries  = []   # global_step at the end of each epoch
epoch_history     = []   # dict per epoch for artifact graphs
global_step       = 0
best_composite    = -float("inf")
best_val_loss     = float("inf")

best_ckpt_dir = Path(CONFIG["output_dir"]) / "best_checkpoint"
best_ckpt_dir.mkdir(parents=True, exist_ok=True)

# Guard: ensure clip_projector params are in the optimizer (not double-added).
# Section 12 adds them explicitly; this is a safety check for re-running cells.
projector_param_ids = {id(p) for p in clip_projector.parameters() if p.requires_grad}
opt_param_ids       = {id(p) for g in optimizer.param_groups for p in g["params"]}
new_proj_params     = [p for p in clip_projector.parameters()
                       if p.requires_grad and id(p) not in opt_param_ids]
if new_proj_params:
    optimizer.add_param_group({"params": new_proj_params, "lr": CONFIG["lr"]})
    print(f"Added {len(new_proj_params)} CLIP-projector tensors to optimizer.")

# TF32 matmuls prevent fp16 overflow in attention layers on T4.
torch.backends.cuda.matmul.fp32_precision = "tf32"
torch.backends.cudnn.conv.fp32_precision  = "tf32"

# Conservative GradScaler init (2^14 vs default 2^16) to reduce early overflow.
scaler = torch.amp.GradScaler("cuda", init_scale=2**14)

# Rolling cache of last-computed expensive text metrics (refreshed periodically).
latest_text_metrics = {
    "cider": np.nan, "meteor": np.nan,
    "bleu1": np.nan, "bleu2": np.nan,
    "bleu3": np.nan, "bleu4": np.nan,
    "rouge": np.nan,
}

model.train()

for epoch in range(1, CONFIG["epochs"] + 1):
    epoch_start = time.time()

    # ── Training ──────────────────────────────────────────────────────────────
    model.train()
    clip_projector.train()
    train_ce_sum       = 0.0
    train_clip_aux_sum = 0.0
    train_total_sum    = 0.0
    train_steps        = 0

    # Linear warm-in of CLIP aux weight over the first clip_aux_warmup_epochs.
    alpha_scale = min(1.0, epoch / max(CONFIG["clip_aux_warmup_epochs"], 1))
    alpha       = CONFIG["clip_aux_weight"] * alpha_scale

    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{CONFIG['epochs']} [train]", leave=True)
    for batch in pbar:
        batch = {k: v.to(device) for k, v in batch.items()}
        if "pixel_values" in batch:
            batch["pixel_values"] = batch["pixel_values"].to(torch.float16)

        optimizer.zero_grad()

        with torch.amp.autocast("cuda", dtype=torch.float16):
            # output_hidden_states=True exposes decoder_hidden_states for CLIP aux.
            outputs = model(**batch, output_hidden_states=True, return_dict=True)
            ce_loss = outputs.loss

            if (global_step % max(CONFIG["clip_aux_every_n_steps"], 1)) == 0:
                clip_aux_loss = compute_clip_aux_loss(outputs, batch["pixel_values"], processor)
            else:
                clip_aux_loss = torch.tensor(0.0, device=ce_loss.device, dtype=ce_loss.dtype)

            total_loss = ce_loss + alpha * clip_aux_loss

        scaler.scale(total_loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        ce_val    = ce_loss.item()
        clip_val  = float(clip_aux_loss.item())
        total_val = total_loss.item()

        # Skip NaN/inf steps — GradScaler already skips the optimizer update.
        if not np.isfinite(total_val):
            global_step += 1
            pbar.set_postfix({"loss": "NaN-skip"})
            continue

        train_ce_sum       += ce_val
        train_clip_aux_sum += clip_val
        train_total_sum    += total_val
        train_steps        += 1
        step_train_losses.append((global_step, total_val))
        global_step += 1

        pbar.set_postfix({
            "ce":       f"{ce_val:.4f}",
            "clip_aux": f"{clip_val:.4f}",
            "total":    f"{total_val:.4f}",
        })

    avg_train_ce       = train_ce_sum       / max(train_steps, 1)
    avg_train_clip_aux = train_clip_aux_sum / max(train_steps, 1)
    avg_train_total    = train_total_sum    / max(train_steps, 1)
    epoch_boundaries.append(global_step)

    # ── Validation CE loss ────────────────────────────────────────────────────
    model.eval()
    clip_projector.eval()
    val_loss_sum = 0.0
    val_steps    = 0

    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f"Epoch {epoch}/{CONFIG['epochs']} [val-ce]", leave=False):
            batch = {k: v.to(device) for k, v in batch.items()}
            if "pixel_values" in batch:
                batch["pixel_values"] = batch["pixel_values"].to(torch.float16)
            with torch.amp.autocast("cuda", dtype=torch.float16):
                outputs = model(**batch)
            v_loss = outputs.loss.item()
            if np.isfinite(v_loss):
                val_loss_sum += v_loss
                val_steps    += 1

    avg_val_loss = val_loss_sum / max(val_steps, 1)
    epoch_val_losses.append(avg_val_loss)

    # ── Metric-aware validation (uses original UICD ground-truth refs) ────────
    val_subset = _deterministic_subset(
        val_set,
        max_items=CONFIG["metrics_subset_size"],
        seed=CONFIG["seed"],
    )

    val_clip = compute_clip_similarity_on_samples(
        model, processor, val_subset,
        max_new_tokens=CONFIG["max_new_tokens_eval"],
        num_beams=CONFIG["num_beams_eval"],
    )

    run_full_metrics = (
        (epoch % CONFIG["full_metrics_every_n_epochs"] == 0)
        or (epoch == CONFIG["epochs"])
    )
    if run_full_metrics:
        latest_text_metrics = compute_text_metrics_on_samples(
            model, processor, val_subset,
            max_new_tokens=CONFIG["max_new_tokens_eval"],
            num_beams=CONFIG["num_beams_eval"],
        )

    metrics_for_score = {
        "clip":   val_clip,
        "cider":  latest_text_metrics.get("cider",  np.nan),
        "meteor": latest_text_metrics.get("meteor", np.nan),
        "bleu4":  latest_text_metrics.get("bleu4",  np.nan),
        "rouge":  latest_text_metrics.get("rouge",  np.nan),
    }
    composite_score = compute_composite_score(
        metrics_for_score,
        weights=CONFIG["metric_weights"],
        norm=CONFIG["metric_norm"],
    )

    epoch_seconds = time.time() - epoch_start
    lr_now    = float(optimizer.param_groups[0]["lr"])
    gpu_mem_gb = float(torch.cuda.max_memory_allocated() / (1024**3)) if torch.cuda.is_available() else 0.0

    epoch_record = {
        "epoch":               epoch,
        "train_ce_loss":       float(avg_train_ce),
        "train_clip_aux_loss": float(avg_train_clip_aux),
        "train_total_loss":    float(avg_train_total),
        "val_ce_loss":         float(avg_val_loss),
        "val_clip":            float(val_clip),
        "cider":               float(latest_text_metrics.get("cider",  np.nan)),
        "meteor":              float(latest_text_metrics.get("meteor", np.nan)),
        "bleu1":               float(latest_text_metrics.get("bleu1",  np.nan)),
        "bleu2":               float(latest_text_metrics.get("bleu2",  np.nan)),
        "bleu3":               float(latest_text_metrics.get("bleu3",  np.nan)),
        "bleu4":               float(latest_text_metrics.get("bleu4",  np.nan)),
        "rouge":               float(latest_text_metrics.get("rouge",  np.nan)),
        "composite_score":     float(composite_score),
        "learning_rate":       lr_now,
        "clip_alpha":          float(alpha),
        "epoch_seconds":       float(epoch_seconds),
        "max_gpu_mem_gb":      gpu_mem_gb,
        "full_metrics_ran":    bool(run_full_metrics),
    }
    epoch_history.append(epoch_record)

    print(
        f"\nEpoch {epoch} | train_ce: {avg_train_ce:.4f} | train_clip_aux: {avg_train_clip_aux:.4f} "
        f"| train_total: {avg_train_total:.4f} | val_ce: {avg_val_loss:.4f} "
        f"| val_clip: {val_clip:.4f} | composite: {composite_score:.4f}"
    )

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss

    # ── Save best checkpoint by composite score ───────────────────────────────
    if composite_score > best_composite:
        best_composite = composite_score
        # Pre-save the base model config so PEFT can verify vocab wasn't modified.
        model.text_decoder.base_model.model.config.save_pretrained(str(best_ckpt_dir))
        model.text_decoder.save_pretrained(str(best_ckpt_dir))
        processor.save_pretrained(str(best_ckpt_dir))
        torch.save(clip_projector.state_dict(), str(best_ckpt_dir / "clip_projector.pt"))
        print(f"  ✓ Best checkpoint saved (composite={best_composite:.4f}, val_ce={avg_val_loss:.4f})")

print("\nTraining complete.")
print(f"Best composite score : {best_composite:.4f}")
print(f"Best val CE (safety) : {best_val_loss:.4f}")


## 14 · Loss Visualisation

In [ ]:
def smooth(values: list, window: int = 50) -> np.ndarray:
    """Simple rolling-mean smoother (same-length output, edge-padded)."""
    arr = np.array(values, dtype=np.float32)
    if len(arr) < window:
        return arr
    kernel = np.ones(window) / window
    return np.convolve(arr, kernel, mode="same")


output_dir = Path(CONFIG["output_dir"])
output_dir.mkdir(parents=True, exist_ok=True)

# ── Persist epoch history ─────────────────────────────────────────────────────
history_path = output_dir / "epoch_history.json"
with open(history_path, "w", encoding="utf-8") as f:
    json.dump(epoch_history, f, indent=2)
print(f"Saved history: {history_path}")

run_metadata = {
    "student_model_id":         CONFIG["student_model_id"],
    "teacher_path":             CONFIG["teacher_path"],
    "seed":                     CONFIG["seed"],
    "split":                    CONFIG["split"],
    "epochs":                   CONFIG["epochs"],
    "batch_size":               CONFIG["batch_size"],
    "lr":                       CONFIG["lr"],
    "lora_r":                   CONFIG["lora_r"],
    "lora_alpha":               CONFIG["lora_alpha"],
    "lora_target_modules":      CONFIG["lora_target_modules"],
    "n_pseudo_captions":        CONFIG["n_pseudo_captions"],
    "clip_aux_weight":          CONFIG["clip_aux_weight"],
    "clip_aux_warmup_epochs":   CONFIG["clip_aux_warmup_epochs"],
    "clip_aux_every_n_steps":   CONFIG["clip_aux_every_n_steps"],
    "metric_weights":           CONFIG["metric_weights"],
    "metric_norm":              CONFIG["metric_norm"],
}
metadata_path = output_dir / "run_metadata.json"
with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(run_metadata, f, indent=2)
print(f"Saved metadata: {metadata_path}")

# ── Graph 1: Step-level and epoch-level losses ────────────────────────────────
steps    = [s for s, _ in step_train_losses]
losses   = [l for _, l in step_train_losses]
smoothed = smooth(losses)

epochs_x       = [r["epoch"]               for r in epoch_history]
train_ce       = [r["train_ce_loss"]        for r in epoch_history]
train_clip_aux = [r["train_clip_aux_loss"]  for r in epoch_history]
train_total    = [r["train_total_loss"]     for r in epoch_history]
val_ce         = [r["val_ce_loss"]          for r in epoch_history]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle("BLIP-1 Student LoRA — Loss Progress", fontsize=14, fontweight="bold")

if steps:
    ax1.plot(steps, losses,   color="steelblue", alpha=0.25, linewidth=0.8, label="Per-step total")
    ax1.plot(steps, smoothed, color="steelblue",             linewidth=1.8, label="Smoothed total")
    for i, boundary in enumerate(epoch_boundaries):
        ax1.axvline(x=boundary, color="gray", linestyle="--", linewidth=0.8,
                    label="Epoch boundary" if i == 0 else None)
ax1.set_xlabel("Training step"); ax1.set_ylabel("Loss")
ax1.set_title("Step-level Total Loss"); ax1.legend(fontsize=9); ax1.grid(True, alpha=0.3)

ax2.plot(epochs_x, train_ce,       color="tab:blue",   marker="o", linewidth=1.6, label="Train CE")
ax2.plot(epochs_x, train_clip_aux, color="tab:purple", marker="o", linewidth=1.6, label="Train CLIP aux")
ax2.plot(epochs_x, train_total,    color="tab:green",  marker="o", linewidth=1.6, label="Train total")
ax2.plot(epochs_x, val_ce,         color="tab:orange", marker="o", linewidth=1.8, label="Val CE")
ax2.set_xlabel("Epoch"); ax2.set_ylabel("Loss")
ax2.set_title("Epoch-level Losses"); ax2.set_xticks(epochs_x)
ax2.legend(fontsize=9); ax2.grid(True, alpha=0.3)

plt.tight_layout()
loss_plot_path = output_dir / "loss_curves.png"
plt.savefig(loss_plot_path, dpi=150, bbox_inches="tight")
print(f"Saved graph: {loss_plot_path}")
plt.show()

# ── Graph 2: Validation metric curves ────────────────────────────────────────
val_clip_hist = [r["val_clip"] for r in epoch_history]
cider         = [r["cider"]    for r in epoch_history]
meteor        = [r["meteor"]   for r in epoch_history]
bleu4         = [r["bleu4"]    for r in epoch_history]
rouge         = [r["rouge"]    for r in epoch_history]

plt.figure(figsize=(12, 6))
plt.plot(epochs_x, val_clip_hist, marker="o", linewidth=2.0, label="CLIP")
plt.plot(epochs_x, cider,         marker="o", linewidth=1.5, label="CIDEr")
plt.plot(epochs_x, meteor,        marker="o", linewidth=1.5, label="METEOR")
plt.plot(epochs_x, bleu4,         marker="o", linewidth=1.5, label="BLEU-4")
plt.plot(epochs_x, rouge,         marker="o", linewidth=1.5, label="ROUGE-L")
plt.xlabel("Epoch"); plt.ylabel("Metric value")
plt.title("Validation Metric Progress (BLIP-1 Student)")
plt.xticks(epochs_x); plt.grid(True, alpha=0.3); plt.legend(ncol=3)
metric_plot_path = output_dir / "metric_curves.png"
plt.savefig(metric_plot_path, dpi=150, bbox_inches="tight")
print(f"Saved graph: {metric_plot_path}")
plt.show()

# ── Graph 3: Composite score trajectory ───────────────────────────────────────
composite      = [r["composite_score"] for r in epoch_history]
best_epoch_idx = int(np.nanargmax(np.array(composite, dtype=np.float32))) if composite else 0
best_epoch     = epochs_x[best_epoch_idx]  if epochs_x  else 0
best_score     = composite[best_epoch_idx] if composite else float("nan")

plt.figure(figsize=(10, 5))
plt.plot(epochs_x, composite, color="purple", marker="o", linewidth=2.0, label="Composite score")
if epochs_x:
    plt.scatter([best_epoch], [best_score], color="red", s=80, label=f"Best epoch={best_epoch}")
plt.xlabel("Epoch"); plt.ylabel("Composite score")
plt.title("Checkpoint Selection Score Progress"); plt.xticks(epochs_x)
plt.grid(True, alpha=0.3); plt.legend()
composite_plot_path = output_dir / "composite_score_curve.png"
plt.savefig(composite_plot_path, dpi=150, bbox_inches="tight")
print(f"Saved graph: {composite_plot_path}")
plt.show()

# ── Graph 4: Efficiency trend ─────────────────────────────────────────────────
epoch_seconds_hist = [r["epoch_seconds"]  for r in epoch_history]
gpu_mem_hist       = [r["max_gpu_mem_gb"] for r in epoch_history]

fig, ax1 = plt.subplots(figsize=(11, 5))
ax1.plot(epochs_x, epoch_seconds_hist, color="tab:blue",   marker="o", linewidth=1.8, label="Epoch time (s)")
ax1.set_xlabel("Epoch"); ax1.set_ylabel("Seconds", color="tab:blue")
ax1.tick_params(axis="y", labelcolor="tab:blue"); ax1.grid(True, alpha=0.3)

ax2 = ax1.twinx()
ax2.plot(epochs_x, gpu_mem_hist, color="tab:orange", marker="o", linewidth=1.8, label="Max GPU mem (GB)")
ax2.set_ylabel("GB", color="tab:orange"); ax2.tick_params(axis="y", labelcolor="tab:orange")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")
plt.title("Training Efficiency Progress")
eff_plot_path = output_dir / "efficiency_curve.png"
plt.savefig(eff_plot_path, dpi=150, bbox_inches="tight")
print(f"Saved graph: {eff_plot_path}")
plt.show()

## 15 · Save Outputs

Saves all artefacts to `/kaggle/working/blip1_student_uicd/`:

| Path | Contents |
|------|----------|
| `best_checkpoint/` | Best checkpoint by weighted composite score (LoRA adapters + processor + CLIP projector) |
| `lora_adapters/` | Final-epoch LoRA adapter weights (`adapter_model.bin` + config) |
| `clip_projector.pt` | CLIP alignment projector weights (student text hidden → CLIP embed) |
| `processor/` | Tokeniser and image processor |
| `merged_model/` | Full merged model (base + LoRA fused, ~400 M params) |
| `pseudo_labels.json` | Teacher-generated pseudo-labels used for student training |
| `epoch_history.json` | Per-epoch metric and efficiency logs |
| `run_metadata.json` | Training configuration snapshot |
| `loss_curves.png` | Step/epoch loss progression |
| `metric_curves.png` | CLIP/CIDEr/METEOR/BLEU-4/ROUGE trends |
| `composite_score_curve.png` | Weighted checkpoint score trajectory |
| `efficiency_curve.png` | Epoch runtime and max GPU memory trend |

In [ ]:

output_dir = Path(CONFIG["output_dir"])

# ── 1. Final-epoch LoRA adapter weights (text_decoder only) ──────────────────
lora_dir = output_dir / "lora_adapters"
lora_dir.mkdir(parents=True, exist_ok=True)
# Pre-save the base model config so PEFT can verify vocab wasn't modified.
model.text_decoder.base_model.model.config.save_pretrained(str(lora_dir))
model.text_decoder.save_pretrained(str(lora_dir))
print(f"LoRA adapters saved  → {lora_dir}")

# ── 2. CLIP projector ─────────────────────────────────────────────────────────
clip_proj_path = output_dir / "clip_projector.pt"
torch.save(clip_projector.state_dict(), str(clip_proj_path))
print(f"CLIP projector saved → {clip_proj_path}")

# ── 3. Processor ──────────────────────────────────────────────────────────────
proc_dir = output_dir / "processor"
processor.save_pretrained(str(proc_dir))
print(f"Processor saved      → {proc_dir}")

# ── 4. Merged model (base + LoRA fused, ~400 M params) ───────────────────────
print("Merging LoRA weights into base model...")
model.text_decoder = model.text_decoder.merge_and_unload()
merged_dir = output_dir / "merged_model"
model.save_pretrained(str(merged_dir))
processor.save_pretrained(str(merged_dir))   # co-locate processor for easy loading
print(f"Merged model saved   → {merged_dir}")

# ── 5. Summary ────────────────────────────────────────────────────────────────
print("\n── Saved files ─────────────────────────────────────────────────")
for p in sorted(output_dir.rglob("*")):
    if p.is_file():
        size_mb = p.stat().st_size / 1024**2
        print(f"  {p.relative_to(output_dir)}  ({size_mb:.1f} MB)")
print("────────────────────────────────────────────────────────────────")


## 16 · Package Outputs as ZIP

Creates `blip1_student_uicd.zip` in `/kaggle/working/` containing all saved artefacts for easy download.

In [ ]:
import zipfile

output_dir = Path(CONFIG["output_dir"])
zip_path   = output_dir.parent / f"{output_dir.name}.zip"

print(f"Creating archive: {zip_path}")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for file in sorted(output_dir.rglob("*")):
        if file.is_file():
            arcname = file.relative_to(output_dir.parent)
            zf.write(file, arcname)

zip_size_mb = zip_path.stat().st_size / 1024**2
print(f"Done. Archive size: {zip_size_mb:.1f} MB  →  {zip_path}")